In [1]:
import os
import pathlib
import sys

import IPython.display
import polars

JSON2CBOR_EVAL_DIR = pathlib.Path.cwd()
os.environ["JSON2CBOR_EVAL_DIR"] = str(JSON2CBOR_EVAL_DIR)

if str((JSON2CBOR_EVAL_DIR / "..").absolute()) not in sys.path:
    sys.path.append(str((JSON2CBOR_EVAL_DIR / "..").absolute()))

from utils import list_code

# JSON to CBOR: Dataset Collection

**You can also just unpack the `jsons.tar.gz` in `03_json2cbor_eval.tar.xz` on [Zenodo](https://tbd) to get our original dataset.**

```sh
tar -C ./03_json2cbor_eval -I pigz -xf ./03_json2cbor_eval/jsons.tar.gz
```

## Scraping JSON responses from HTTP Archive

We used the following SQL instructions to download from the legacy BigQuery database of the HTTP archive on November 14, 2024 (no longer available):

In [2]:
list_code(JSON2CBOR_EVAL_DIR / "input_datasets" / "big_query.sql")

CREATE TABLE cbor_paper.processed_table
PARTITION BY RANGE_BUCKET(export_id, GENERATE_ARRAY(0, 50, 1))
CLUSTER BY export_id
AS (
  SELECT
  url, req_user_agent, resp_content_length, CAST(FLOOR(50*RAND()) AS INT64) AS export_id
FROM `httparchive.summary_requests.2024_09_01_desktop`
WHERE 
  LOWER(resp_content_type) LIKE "%json%"
);
SELECT * EXCEPT(export_id)
FROM cbor_paper.processed_table
WHERE export_id IN (1, 2, 3);
SELECT * EXCEPT(export_id)
FROM cbor_paper.processed_table
WHERE export_id IN (4, 5, 6);

The last two `SELECT`s resulted in the two CSV files provided in `./03_json2cbor_eval/input_datasets` in `03_json2cbor_eval.tar.xz` on [Zenodo](https://tbd).

In [3]:
df = polars.scan_csv(
    f"{JSON2CBOR_EVAL_DIR / "input_datasets"}/*.csv"
).collect()
IPython.display.display(df)
del df

url,req_user_agent,resp_content_length
str,str,str
"""https://px.ads.linkedin.com/at…","""Mozilla/5.0 (X11; Linux x86_64…",null
"""https://lb.eu-1-id5-sync.com/l…","""Mozilla/5.0 (X11; Linux x86_64…",null
"""https://ppsnco.com/?wc-ajax=ge…","""Mozilla/5.0 (X11; Linux x86_64…",null
"""https://siteassets.parastorage…","""Mozilla/5.0 (X11; Linux x86_64…",null
"""https://data.covidactnow.org/s…","""Mozilla/5.0 (X11; Linux x86_64…",null
…,…,…
"""https://static.parastorage.com…","""Mozilla/5.0 (X11; Linux x86_64…","""154"""
"""https://static.parastorage.com…","""Mozilla/5.0 (X11; Linux x86_64…","""154"""
"""https://static.parastorage.com…","""Mozilla/5.0 (X11; Linux x86_64…","""154"""


The information in those files, we used to download the json files by running the `harch_dl.sh` script between November 15, 2024 and Dezember 23, 2024 (UTC). It dumps all parsable JSON files downloaded using the URL and User-Agent in the CSVs in a minified format (stripped of all non-semantic whitespaces) in `./03_json2cbor_eval/jsons/harch/` and provides a summary CSV (which we did not preserve in our Zenodo dataset) `harch_jsons.csv` containing the URL from the original CSV and the filename in `./03_json2cbor_eval/jsons/harch/` relative to `./03_json2cbor_eval/`.

In [4]:
list_code(JSON2CBOR_EVAL_DIR / "harch_dl.sh")

#!/usr/bin/env bash
#
# Copyright (C) 2024-26 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

export INPUT_PATH="${INPUT_PATH:-${SCRIPT_DIR}/input_datasets}"
HARCH_JSONS="${SCRIPT_DIR}/harch_jsons.csv"

cat "${INPUT_PATH}"/bq-results-*.csv | \
    awk -F, 'BEGIN { \
      FPAT = "([^,]+)|(\"[^\"]+\")"; \
      OFS="\t" \
    } {print $1, $2}' | \
    while IFS=$'\t' read url user_agent; do
        [ -f "${HARCH_JSONS}" ] && grep -q -F "${url}" "${HARCH_JSONS}" || \
        curl -s -A "${user_agent}" "${url}" | "${SCRIPT_DIR}"/harch_dl.py "${url}" \
        >> "${HARCH_JSONS}"
    done

In [5]:
list_code(JSON2CBOR_EVAL_DIR / "harch_dl.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2024-26 TU Dresden
#
# Distributed under terms of the MIT license.

import sys
import json
import pathlib
import os
import urllib.parse

SCRIPT_DIR = pathlib.Path(os.path.dirname(os.path.realpath(__file__)))

OUTPUT_PATH = pathlib.Path(
    os.environ.get("OUTPUT_PATH", SCRIPT_DIR / "jsons" / "harch")
)

if __name__ == "__main__":
    try:
        obj = json.loads(sys.stdin.read())
    except (json.JSONDecodeError, UnicodeDecodeError):
        sys.exit(1)
    url = sys.argv[1]
    url_path = urllib.parse.urlparse(sys.argv[1]).path
    if not url_path.endswith(".json"):
        url_path = f"{url_path}.json"
    path = OUTPUT_PATH / pathlib.Path(url_path.lstrip("/"))
    if path.name == ".json":
        path = path.parent / "index.json"
    if len(path.name) > 255:
        path = path.parent / f"{path.name[:246]}.json"
    counter = 0
    while path.exists():
        if counter == 0:
            path = path.parent / f"{'.'.join(path.name.split('.')[:-1])}.{counter:003d}.json"
        else:
            path = path.parent / f"{'.'.join(path.name.split('.')[:-2])}.{counter:003d}.json"
        counter += 1
    if not path.parent.exists():
        path.parent.mkdir(parents=True)
    with open(path, "w") as json_file:
        json.dump(obj, json_file, separators=(",", ":"))
    print(url, path.relative_to(OUTPUT_PATH), sep="\t")

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${JSON2CBOR_EVAL_DIR}'/../.env/bin/activate;` before the `${JSON2CBOR_EVAL_DIR}/harch_dl.sh`):

In [ ]:
%%bash
tmux new-session -s "harch_dl" -d "'${JSON2CBOR_EVAL_DIR}'/harch_dl.sh"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "harch_dl"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "harch_dl" C-c
```

## Scraping JSON files from GitHub

**Attention: Due to the rate-limiting imposed by GitHub, it is recommended to run this script for at least a month, to get the same results we provide on [Zenodo](https://tbd).**
This collects the GitHub-based JSONs we provide provide on [Zenodo](https://tbd) `./03_json2cbor_eval/jsons/github` (when all tars are unpacked). Our original scrape was conducted from January 15, 2024 to 

### Generate a Fine-Grained GitHub Personal Access Token

First, you need to generate a [fine-grained token](https://github.com/settings/tokens?type=beta) by clicking on “Generate New Token”. Public Repository (read-only) access is sufficient, set the expiration to e.g. 90 days:

<img src="./figs/fine-grained-token.png" alt="Example form for a fine-grained token" border="1" />

Store the resulting token in the file `./03_json2cbor_eval/.gh_token`.

### Run the Scraping Script
Use the following script to then scrape the GitHub repositories for JSON files.

In [7]:
list_code(JSON2CBOR_EVAL_DIR / "scrape_github_repos.py")

#! /usr/bin/env python3
#
# Copyright (C) 2023-26 TU Dresden
#
# Distributed under terms of the MIT license.

import base64
import json
import http.client
import pathlib
import pprint
import re
import os
import time
import urllib.parse

import agithub.GitHub
import json5

SCRIPT_PATH = pathlib.Path(__file__).resolve().parent
OUTPUT_PATH = pathlib.Path(os.environ.get("OUTPUT_PATH", SCRIPT_PATH)).resolve()
GITHUB_PATH = OUTPUT_PATH / "jsons" / "github"
PER_PAGE = 100


class JSON5DecodeError(Exception):
    pass


class AlreadyDownloaded(Exception):
    pass


def page_ref_by_rel(headers, rel):
    try:
        match = re.search(rf"<([^>]+)>; rel=\"{rel}\"", headers["Link"])
    except KeyError:
        return None
    if not match:
        return None
    url = urllib.parse.urlparse(match[1])
    queries = urllib.parse.parse_qs(url.query)
    queries = {k: v[0] for k, v in queries.items()}
    return queries


def page_ref(headers):
    return page_ref_by_rel(headers, "next")


def prepare():
    with open(SCRIPT_PATH / ".gh_token") as token_file:
        token = token_file.readline().strip()
    return agithub.GitHub.GitHub(token=token)


def store_json(path: pathlib.Path, name: str, obj: dict | list):
    if not path.exists():
        path.mkdir(parents=True)
    try:
        with open(path / name, "w", encoding="utf-8") as json_file:
            json.dump(obj, json_file, ensure_ascii=False, separators=(",", ":"))
    except UnicodeEncodeError:
        (path / name).unlink()
        raise


def default_queries(queries):
    return {
        "page": queries.get("page", 1),
        "per_page": queries.get("per_page", 30),
        "since": queries.get("since", 0),
    }


def users(github):
    queries = {"per_page": PER_PAGE}
    while queries is not None:
        status = 500
        while status // 100 == 5 or status == 408:
            try:
                status, users = github.users.get(**queries)
            except (ConnectionResetError, TimeoutError):
                time.sleep(5)
                continue
            if status // 100 == 5 or status == 408:
                time.sleep(10)
            elif status != 200:
                raise http.client.HTTPException(
                    f"Error status {status} for users {queries}"
                )
        store_json(
            GITHUB_PATH / "github" / "users",
            "page{page}_since{since}.json".format(**default_queries(queries)),
            users,
        )
        queries = page_ref(dict(github.getheaders()))
        for user in users:
            yield user


def repos(github, user):
    queries = {"per_page": PER_PAGE}
    while queries is not None:
        status = 500
        while status // 100 == 5 or status == 408:
            try:
                status, repos = github.get(url=user["repos_url"], **queries)
            except (ConnectionResetError, TimeoutError):
                time.sleep(5)
                continue
            if status // 100 == 5 or status == 408:
                time.sleep(10)
            elif status != 200:
                raise http.client.HTTPException(
                    f"Error status {status} for repos {queries}"
                )
        if repos:
            store_json(
                GITHUB_PATH / "github" / user["login"],
                "page{page}_since{since}.json".format(**default_queries(queries)),
                repos,
            )
        queries = page_ref(dict(github.getheaders()))
        for repo in repos:
            yield repo


def get_git_blob(github, item, sha_filenames):
    blobs_path = GITHUB_PATH / "github" / "blobs"
    org, repo = item["repository"]["full_name"].split("/")
    name = item["name"]
    path = item["path"].split("/")
    json_path = GITHUB_PATH / org / repo / pathlib.Path(*path[:-1])
    sha = item["sha"]
    print(
        sha,
        str((json_path / name).relative_to(OUTPUT_PATH)),
        sep=";",
        file=sha_filenames
    )
    if (blobs_path / f"{sha}.json").exists():
 

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${JSON2CBOR_EVAL_DIR}'/../.env/bin/activate;` before the `${JSON2CBOR_EVAL_DIR}/scrape_github_repos.py`):

In [ ]:
%%bash
tmux new-session -s "scrape_github_repos" -d "${JSON2CBOR_EVAL_DIR}/scrape_github_repos.py"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "scrape_github_repos"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "scrape_github_repos" C-c
```

## Conversion to CBOR

To make our runtime recreatable, we decided to do the conversion of the GitHub dataset on a Raspberry Pi. The results for that can be found in `./03_json2cbor_eval/json2cbor_rpi3.csv.gz` in the `03_json2cbor_eval.tar.xz` on [Zenodo](https://tbd).
You can also run the conversion natively, e.g., in the Dockerized setup. We did that for both the GitHub dataset (with `ITERATIONS=1000`) and or the HTTP Archive (with `ITERATIONS=10000`) dataset on a machine with dual AMD EPYC 7702 CPU. The results for that can be found in `./03_json2cbor_eval/json2cbor_epyc7702.csv.gz` in the `03_json2cbor_eval.tar.xz` on [Zenodo](https://tbd).
The resulting CBORs can be found in `./03_json2cbor_eval/cbors.tar.gz` there as well. You will need those unpacked along with the `json2cbor_*.csv.gz` (which can be used as is):

```sh
tar -C 03_json2cbor_eval -I pigz -xf ./cbors.tar.gz
```

### Set-up and Running on Raspberry Pi

For the Raspberry Pi set-up follow the following set-up instructions:

- Get your hands on a [Raspberry Pi 3B+](https://www.raspberrypi.com/products/raspberry-pi-3-model-b-plus/).
- Use the [Raspberry Pi Imager](https://github.com/raspberrypi/rpi-imager) to install [Raspberry Pi OS Lite 64-bit bookworm](https://downloads.raspberrypi.com/raspios_lite_arm64/images/) (or newer, but we ran the original conversion on bookworm) on a Micro SD card
- [Set-up WiFi and SSH](https://www.raspberrypi.com/documentation/computers/configuration.html#setting-up-a-headless-raspberry-pi), if necessary.
- Boot the Raspberry Pi with the Micro SD card inserted and log-in, e.g. via SSH:
  ```sh
  ssh "<username>@<raspi-addr-or-hostname>"
  ```
- Upgrade software and install dependencies:
  ```sh
  sudo apt update && sudo apt upgrade
  sudo apt install tmux git npm python3 \
    libbz2-dev libffi-dev libgdbm-dev libgdbm-compat-dev liblzma-dev \
    libncurses5-dev libreadline-dev libsqlite3-dev libssl-dev \
    lzma lzma-dev parallel pigz tk-dev uuid-dev zlib1g-dev
  ```
- Increase swap file to 1024 GBytes:
  ```sh
  sudo dphys-swapfile swapoff
  sudo nano /etc/dphys-swapfile
  ```
  
  ---
  
  ```sh
  # ...
  CONF_SWAPSIZE=1024
  # ...
  ```
  
  ---
  
  ```sh
  sudo dphys-swapfile setup
  sudo dphys-swapfile swapon
  ```
- Clone artifacts:
  ```sh
  git clone https://tbd cbor-dns-eval-tbd
  cd cbor-dns-eval-tbd
  ```
- Setup Python and install Python and node.js requirments:
  ```sh
  sudo pip install --break-system-packages uv
  uv python install cpython-3.12
  uv venv --python python3.12 .env
  . .env/bin/activate
  uv pip install -r requirements.txt
  ( cd 03_json2cbor_eval/; npm install @sourcemeta/json-taxonomy@1.1.1 )
  ```
- Reboot Raspberry Pi and re-login:
  ```sh
  sudo reboot
  ```
  
  ---
  
  ```sh
  ssh "<username>@<raspi-addr-or-hostname>"
  ```
- Run (ideally in [screen] or [tmux]):
  ```
  cd ~/cbor-dns-eval-tbd/03_json2cbor_eval/
  . ../.env/bin/activate
  ITERATIONS=100 ./json2cbor.sh json2cbor_rpi3.csv
  pigz json2cbor_rpi3.csv
  ```
  **This will override the `03_json2cbor_eval/json2cbor_rpi.csv.gz` on your Raspberry Pi.**

You can find the code for `json2cbor.sh` [below](#Conversion-Code).

[screen]: https://www.gnu.org/software/screen/
[tmux]: https://github.com/tmux/tmux

### Running Natively

Assuming the dockerized set-up, you can, e.g., start the script like this in a `tmux` session. First, install the `node.js`-based `json-taxonomy` package (if you run in an UV-based setup you might need to install `npm` first).

In [9]:
!npm install @sourcemeta/json-taxonomy@1.1.1

m##################) ⠙ reify:@sourcemeta/json-taxonomy: http fetch GET 200 httple
added 1 package in 2s


Then, e.g., start it like this in a `tmux` session with the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${JSON2CBOR_EVAL_DIR}'/../.env/bin/activate;` before the `${JSON2CBOR_EVAL_DIR}/json2cbor.sh`). **Only run this when the TMUX sessions `"harch_dl"` and `"scrape_github_repos"` are _not_ running.** Otherwise, newly added files by those scripts might not be included. If there are no files in `./03_json2cbor_eval/jsons`, this command will have no effect.

In [ ]:
%%bash
tmux new-session -s "json2cbor" -d "'${JSON2CBOR_EVAL_DIR}/json2cbor.sh' '${JSON2CBOR_EVAL_DIR}/json2cbor_native.csv'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "json2cbor"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "json2cbor" C-c
```

#### Our Dataset

We constructed our natively run dataset as follows (in a TMUX session):

```sh
JSON_DATASET=github ITERATIONS=1000 ./json2cbor.sh json2cbor_epyc7702_raw.csv
JSON_DATASET=harch ITERATIONS=10000 ./json2cbor.sh harch_json2cbor_epyc7702.csv
tail -n +2 harch_json2cbor_epyc7702.csv >> json2cbor_epyc7702_raw.csv
```

to get to the `json2cbor_epyc7702.csv.gz` you find on Zenodo, see ["Missing Taxonomy"](#Missing-Taxonomy)

### Conversion Code

Below you can see the code for `./03_json2cbor_eval/json2cbor.sh` as well as the Python script `./03_json2cbor_eval/json2cbor.py` that the `.sh` script calls for each file.

In [11]:
list_code(JSON2CBOR_EVAL_DIR / "json2cbor.sh")

#!/usr/bin/env bash
#
# Copyright (C) 2024-26 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

PROCS=$(grep -c '^processor' /proc/cpuinfo)
if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi
export INPUT_PATH="${INPUT_PATH:-${SCRIPT_DIR}/jsons}"
export OUTPUT_PATH="${OUTPUT_PATH:-${SCRIPT_DIR}/cbors}"

if [ $# -lt 1 ]; then
    echo "usage: $0 <output file>" >&2
    exit 1
fi

export OUTPUT_FILE="$(readlink -f "${1}")"
export JSON_DATASET="${JSON_DATASET:-}"

"${SCRIPT_DIR}"/json2cbor.py --header > "${OUTPUT_FILE}"
find "${INPUT_PATH}" -type f | \
    parallel --line-buffer -j"${PROCS}" -I'{}' \
        "${SCRIPT_DIR}"/json2cbor.py '{}' >> "${OUTPUT_FILE}"

In [12]:
list_code(JSON2CBOR_EVAL_DIR / "json2cbor.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2023-26 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import base64
import gzip
import json
import pathlib
import os
import re
import subprocess
import sys
import timeit
import traceback

import cbor2
import json5

SCRIPT_DIR = pathlib.Path(os.path.dirname(os.path.realpath(__file__)))

INPUT_PATH = pathlib.Path(os.environ.get("INPUT_PATH", SCRIPT_DIR / "jsons"))
OUTPUT_PATH = pathlib.Path(os.environ.get("OUTPUT_PATH", SCRIPT_DIR / "cbors"))
JSON_TAXONOMY_PATH = pathlib.Path(os.environ.get("JSON_TAXONOMY_PATH", SCRIPT_DIR / "node_modules" / ".bin" / "json-taxonomy"))
GITHUB_PATH = INPUT_PATH / "github" / "github"
GITHUB_BLOBS_PATH = GITHUB_PATH / "blobs"
GITHUB_USERS_PATH = GITHUB_PATH / "users"
JSON_DATASET = os.environ.get("JSON_DATASET")
ITERATIONS = int(os.environ.get("ITERATIONS", 1000))


def output_path(json_filename, suffix):
    cbor_filename = OUTPUT_PATH / json_filename.relative_to(
        INPUT_PATH
    ).with_suffix("").with_suffix(suffix)
    cbor_filename.parent.mkdir(parents=True, exist_ok=True)
    return cbor_filename


def encode_cbor(json_filename, json_obj):
    # mark as encoded CBOR data item (see https://www.iana.org/assignments/cbor-tags/cbor-tags.xhtml)
    try:
        json_obj["content"] = cbor2.CBORTag(
            24, cbor2.dumps(json5.loads(json_obj["content"]), canonical=True)
        )

        cbor_filename = output_path(json_filename, ".cbor.cbor")
        with open(cbor_filename, "wb") as cbor_file:
            cbor2.dump(json_obj, cbor_file)
            size = cbor_file.tell()
        cbor_file.close()
        return size
    except (RecursionError, ValueError):
        return ""


def encode_binary(json_filename, json_obj):
    size = json_obj.pop("size")
    # mark as embedded JSON (see https://www.iana.org/assignments/cbor-tags/cbor-tags.xhtml)
    json_obj["content"] = base64.b64decode(json_obj["content"].value)

    assert len(json_obj["content"]) == size
    cbor_filename = output_path(json_filename, ".bin.cbor")
    with open(cbor_filename, "wb") as cbor_file:
        cbor2.dump(json_obj, cbor_file)
        return cbor_file.tell()


def tag_base64(json_filename, json_obj):
    encoding = json_obj.pop("encoding")
    assert encoding == "base64"
    # mark as base64 encoded (see https://www.iana.org/assignments/cbor-tags/cbor-tags.xhtml)
    json_obj["content"] = cbor2.CBORTag(34, json_obj["content"])

    cbor_filename = output_path(json_filename, ".b64tag.cbor")
    with open(cbor_filename, "wb") as cbor_file:
        cbor2.dump(json_obj, cbor_file)
        return cbor_file.tell()


def json_walk(json_obj):
    if isinstance(json_obj, dict):
        yield json_obj
        for key, value in json_obj.items():
            yield json_walk(key)
            yield json_walk(value)
    elif isinstance(json_obj, list):
        yield json_obj
        for value in json_obj:
            yield json_walk(value)
    elif isinstance(json_obj, cbor2.CBORTag):
        yield json_obj
        yield json_obj.value
    else:
        yield json_obj


def relative_to(json_filename, other):
    try:
        return isinstance(json_filename.relative_to(other), pathlib.Path)
    except ValueError:
        return False


def json_taxonomy(json_filename):
    if JSON_TAXONOMY_PATH.exists():
        try:
            out = subprocess.check_output([JSON_TAXONOMY_PATH, json_filename], text=True)
            tier, content_type, redundancy, structure = out.strip().split(", ")
            return int(tier.split()[1]), content_type, redundancy, structure
        except subprocess.CalledProcessError as exc:
            print(f"Cannot determine taxonony for {json_filename}", file=sys.stderr)
            print(traceback.format_exc(), file=sys.stderr)
    return "", "", "", ""


def compress(in_filename, out_filename):
    # gzip.open does not work with `with` in python 3.8.
    out_file = gzip.open(out_filename, "wb")
    try:
        wi

### Missing Taxonomy

In some cases the original `node.js` taxonomy tool could not read the JSON files (e.g., because it is not legal JSON for the JavaScript parser but it is for the Python parser, but we also were not able to get the `node.js` script running on the EPYC 7702 machine in 2024, due to an outdated operating system version on that machine). For these cases (and to determine CBOR taxonomies later on), we provided our own python-based tool, which can be called using `parallel` with the following script. The script also excludes all JSONs that are just plain strings. We kept the Raspberry Pi analysis unchanged for some introspection on taxonym determination problems in [`./03_json2cbor_eval.ipynb`](../03_json2cbor_eval.ipynb#Taxonomy-Determination-Problems-on-GitHub).

In [13]:
list_code(JSON2CBOR_EVAL_DIR / "taxonomy.sh")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


#!/usr/bin/env bash
#
# Copyright (C) 2025-26 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

if [ $# -lt 1 ]; then
    echo "usage: $0 <input file>" >&2
    exit 1
fi

PROCS=$(grep -c '^processor' /proc/cpuinfo)
if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi


taxonomy() {
    JSON_FILENAME="$(echo "$1" | cut -d';' -f1)"
    if ! [ -f "${SCRIPT_DIR}/jsons/${JSON_FILENAME}" ]; then
        echo "$1"
        return
    fi
    if ! echo "$1" | grep -q 'stored_json;;;;;'; then
        echo "$1"
        return
    fi
    TAX=$("${SCRIPT_DIR}/../utils/taxonomy.py" "${SCRIPT_DIR}/jsons/${JSON_FILENAME}" | \
        sed "s/\['tier \([0-9]\+\)', '\(.\)\(.*\)', '\(.\)\(.*\)', '\(.\)\(.*\)'\]/\1;\U\2\E\3;\U\4\E\5;\U\6\E\7/g")
    if [ "${TAX}" = "['str', 'str', 'str', 'str']" ]; then
        echo "${TAX}" >&2
        return
    fi
    echo "$1" | sed "s/stored_json;;;;;/stored_json;${TAX};/"
}

INPUT_FILE="${1}"

export -f taxonomy
export SCRIPT_DIR

cat "${INPUT_FILE}" | parallel --line-buffer -j"${PROCS}" -I'{}' taxonomy

In [14]:
list_code(JSON2CBOR_EVAL_DIR / ".." / "utils" / "taxonomy.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2023 TU Dresden
#
# Distributed under terms of the MIT license.


import argparse
import base64
import functools
import itertools
import json

import cbor2


def values(obj):
    if isinstance(obj, dict):
        return list(obj.values())
    elif isinstance(obj, list):
        return obj
    else:
        []


class EncodeCBORDict(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, bytes):
            return base64.b64encode(obj).decode()
        elif isinstance(obj, cbor2.CBORTag):
            return obj.value
        return super().default(obj)


def utf8size(obj):
    return len(json.dumps(obj, ensure_ascii=False, separators=(",", ":"), cls=EncodeCBORDict))


def byte_size(obj):
    if isinstance(obj, (dict, list)):
        return functools.reduce(
            lambda acc, size: {
                "scalar": acc["scalar"] + size["scalar"],
                "structural": acc["structural"] + size["structural"],
            },
            list(
                map(
                    byte_size,
                    values(obj),
                )
            )
            + [
                {
                    "scalar": 0,
                    "structural": (
                        (2 + max(len(obj), 1) - 1)
                        if isinstance(obj, list)
                        else len(obj.keys()) + utf8size(list(obj.keys()))
                    ),
                }
            ],
            {"scalar": 0, "structural": 0},
        )
    return {"scalar": utf8size(obj), "structural": 0}


def deep_values(obj):
    if isinstance(obj, (dict, list)):
        return functools.reduce(
            lambda acc, el: acc + deep_values(el), values(obj), [obj]
        )
    return [obj]


def height(obj):
    if isinstance(obj, (dict, list)):
        return 1 + max([0] + list(map(height, values(obj))))
    return 0


def level(obj, lvl):
    if lvl == 0:
        return [obj]
    elif isinstance(obj, (dict, list)):
        if lvl <= 1:
            return list(
                filter(
                    lambda el: not isinstance(el, (dict, list)),
                    values(obj),
                )
            )
        else:
            return functools.reduce(
                lambda acc, el: acc + level(el, lvl - 1),
                values(obj),
                [],
            )
    else:
        return []


def acc_byte_size(elements, size_keys=["scalar", "structural"]):
    return functools.reduce(
        lambda acc, size: acc + sum(size[k] for k in size_keys),
        map(lambda el: byte_size(el), elements),
        0,
    )


def level_analyze(obj, lvl):
    elements = level(obj, lvl)
    return {
        "count": len(elements),
        "size": acc_byte_size(elements),
    }


def is_deep_equal(left, right):
    if isinstance(left, cbor2.CBORTag):
        if not isinstance(right, cbor2.CBORTag):
            return False
        return left == right
    if isinstance(left, dict):
        if not isinstance(right, dict):
            return False
        elif len(left) != len(right):
            return False

        for key in left:
            if key not in right or not is_deep_equal(left[key], right[key]):
                return False

        return True
    elif isinstance(left, list):
        if not isinstance(right, list):
            return False
        elif len(left) != len(right):
            return False

        for index, value in enumerate(left):
            if not is_deep_equal(value, right[index]):
                return False

        return True
    return left == right


def unique_deep(objs):
    return functools.reduce(
        lambda acc, el: (
            acc + [el] if not any(is_deep_equal(item, el) for item in acc) else acc
        ),
        objs,
        []
    )


def analyze(obj):
    byte_sz = byte_size(obj)
    values = deep_values(obj)
    higt = height(obj)

    textual = list(filter(lambda el: isinstance(el, str), values)

For our natively generated dataset [above](#Our-Dataset), we ran the following command.

In [ ]:
%%bash
tmux new-session -s "taxonomy" -d "'${JSON2CBOR_EVAL_DIR}/taxonomy.sh' '${JSON2CBOR_EVAL_DIR}/json2cbor_epyc7702_raw.csv' > '${JSON2CBOR_EVAL_DIR}/json2cbor_epyc7702.csv'; pigz '${JSON2CBOR_EVAL_DIR}/json2cbor_epyc7702.csv'"